In [6]:
### 1. generate_dataset.py
import pandas as pd, random
restaurants=['Punjabi Kitchen','Imperfecto','Oasis Cafe','Grill and Chill','Flavour of Spice','Relax Point']
positive_templates=[
 'Amazing food at {r}','Loved the ambience at {r}','Excellent service at {r}','Best momos at {r}','Highly recommend {r}'
]
negative_templates=[
 'Terrible service at {r}','Bad hygiene at {r}','Food was cold at {r}','Very disappointing meal at {r}','Overpriced and poor quality at {r}'
]
neutral_templates=[
 'Food was okay at {r}','Average experience at {r}','Nothing special at {r}','Decent place for lunch at {r}','Service was normal at {r}'
]
rows=[]
for _ in range(170): rows.append((random.choice(positive_templates).format(r=random.choice(restaurants)),'positive'))
for _ in range(165): rows.append((random.choice(negative_templates).format(r=random.choice(restaurants)),'negative'))
for _ in range(165): rows.append((random.choice(neutral_templates).format(r=random.choice(restaurants)),'neutral'))
random.shuffle(rows)
pd.DataFrame(rows,columns=['review','sentiment']).to_csv('majitar_reviews_500.csv',index=False)
print('Created majitar_reviews_500.csv with',len(rows),'rows')
### 2. train_and_evaluate.py
import pandas as pd, pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
# load data
df=pd.read_csv('majitar_reviews_500.csv')
X_train,X_test,y_train,y_test=train_test_split(df.review,df.sentiment,test_size=0.2,random_state=42,stratify=df.sentiment)
vec=TfidfVectorizer(stop_words='english',ngram_range=(1,2))
Xtr=vec.fit_transform(X_train)
Xte=vec.transform(X_test)
models={
 'Logistic Regression': LogisticRegression(max_iter=1000),
 'Naive Bayes': MultinomialNB(),
 'SVM': LinearSVC()
}
best_name,best_model,best_acc=None,None,0
for name,m in models.items():
    m.fit(Xtr,y_train)
    pred=m.predict(Xte)
    acc=accuracy_score(y_test,pred)
    print('\n',name,'Accuracy:',round(acc,4))
    print(classification_report(y_test,pred))
    print(confusion_matrix(y_test,pred))
    if acc>best_acc:
        best_acc=acc; best_name=name; best_model=m
pickle.dump((best_model,vec),open('best_sentiment_model.pkl','wb'))
print('Best model:',best_name,best_acc)
### 3. app.py
import streamlit as st, pickle
model,vec=pickle.load(open('best_sentiment_model.pkl','rb'))
st.title('Majitar Restaurant Sentiment Analyzer')
text=st.text_area('Enter review')
if st.button('Analyze'):
    pred=model.predict(vec.transform([text]))[0]
    st.write('Prediction:',pred)

Created majitar_reviews_500.csv with 500 rows

 Logistic Regression Accuracy: 1.0
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        33
     neutral       1.00      1.00      1.00        33
    positive       1.00      1.00      1.00        34

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

[[33  0  0]
 [ 0 33  0]
 [ 0  0 34]]

 Naive Bayes Accuracy: 1.0
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        33
     neutral       1.00      1.00      1.00        33
    positive       1.00      1.00      1.00        34

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

[[33  0  0]
 [ 0 33  0]
 [ 0  0 34]]

 SVM Accuracy: 1.0
              precision    recall 

SyntaxError: invalid syntax (1756900273.py, line 1)